In [19]:
import os

import arcpy
import h3
import pandas as pd
from pyspark.sql.functions import pandas_udf
from pyspark.sql.types import ArrayType, DoubleType, StringType
from spark_esri import spark_start, spark_stop

In [2]:
# spark_stop()

config = {"spark.driver.memory": "16G", "spark.executor.memory": "16G"}
spark = spark_start(config)

In [9]:
@pandas_udf(returnType=StringType())
def geo_to_h3(lat: pd.Series, lon: pd.Series) -> pd.Series:
    return pd.Series([h3.geo_to_h3(l1, l2, 7) for l1, l2 in zip(lat, lon)])

In [10]:
@pandas_udf(returnType=ArrayType(ArrayType(DoubleType())))
def h3_to_geo_boundary(s: pd.Series) -> pd.Series:
    def _flip(b):
        l = [(r, l) for l, r in b]
        return l + [l[0]]

    return pd.Series([_flip(h3.h3_to_geo_boundary(_)) for _ in s])

In [11]:
sp_ref = arcpy.SpatialReference(4326)

In [12]:
fields = ["SHAPE@X", "SHAPE@Y"]
with arcpy.da.SearchCursor("Broadcast", fields, spatial_reference=sp_ref) as rows:
    df = (
        spark.createDataFrame(rows, "lon double,lat double")
        .select(geo_to_h3("lat", "lon").alias("h3"))
        .groupBy("h3")
        .count()
        .select(h3_to_geo_boundary("h3").alias("shape"), "count")
    )

In [13]:
ws = "memory"
nm = "H3"

fc = os.path.join(ws, nm)

arcpy.management.Delete(fc)

arcpy.management.CreateFeatureclass(ws, nm, "POLYGON", spatial_reference=sp_ref)
arcpy.management.AddField(fc, "POP", "LONG")

with arcpy.da.InsertCursor(fc, ["SHAPE@", "POP"]) as cursor:
    for row in df.toLocalIterator():
        # print(row["shape"],row["count"])
        cursor.insertRow((row["shape"], row["count"]))